Notebook 2 — Clasificador SVC

CELDA 1 — Instalación

In [2]:
!pip install yfinance ta scikit-learn --quiet

  Preparing metadata (setup.py) ... done


CELDA 2 — Librerías

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

CELDA 3 — Configuración

In [4]:
TICKERS = [
    "FSM",
    "VOLCABC1.LM",
    "ABX.TO",
    "BVN",
    "BHP"
]

ticker = "BVN"

CELDA 4 — Descarga de datos

In [5]:
df = yf.download(
    ticker,
    period="2y",
    progress=False
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.head()

/tmp/ipykernel_4247/1191372851.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(


Price,Close,High,Low,Open,Volume
Date,,,,,
2024-06-20,16.254087,16.654247,16.187395,16.387475,962900
2024-06-21,16.254087,16.854327,16.063537,16.854327,4528400
2024-06-24,16.663773,16.892435,16.330306,16.339834,2005000
2024-06-25,16.273142,16.682829,16.015896,16.663773,1518000
2024-06-26,15.977788,16.263615,15.815819,16.149285,752900


CELDA 5 — Indicadores técnicos

In [6]:
df["SMA20"] = ta.trend.sma_indicator(df["Close"], 20)
df["SMA50"] = ta.trend.sma_indicator(df["Close"], 50)

df["EMA12"] = ta.trend.ema_indicator(df["Close"], 12)
df["EMA26"] = ta.trend.ema_indicator(df["Close"], 26)

df["RSI"] = ta.momentum.rsi(df["Close"], 14)

macd = ta.trend.MACD(df["Close"])
df["MACD"] = macd.macd()

bb = ta.volatility.BollingerBands(df["Close"])

df["BB_High"] = bb.bollinger_hband()
df["BB_Low"] = bb.bollinger_lband()

df["ATR"] = ta.volatility.average_true_range(
    df["High"],
    df["Low"],
    df["Close"]
)

df.dropna(inplace=True)

CELDA 6 — Variable objetivo

In [7]:
df["Retorno"] = df["Close"].pct_change()

df["Trend"] = np.where(
    df["Retorno"].shift(-1) > 0,
    1,
    0
)

df.dropna(inplace=True)

CELDA 7 — Variables predictoras

In [8]:
X = df[
    [
        "SMA20",
        "SMA50",
        "EMA12",
        "EMA26",
        "RSI",
        "MACD",
        "BB_High",
        "BB_Low",
        "ATR"
    ]
]

y = df["Trend"]

CELDA 8 — Escalamiento

In [9]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

CELDA 9 — Train/Test

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    shuffle=False
)

CELDA 10 — GridSearch

In [11]:
parametros = {
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    SVC(),
    parametros,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='accuracy')

CELDA 11 — Mejor modelo

In [12]:
modelo = grid.best_estimator_

pred = modelo.predict(X_test)

CELDA 12 — Métricas

In [13]:
accuracy = accuracy_score(y_test, pred)

precision = precision_score(y_test, pred)

recall = recall_score(y_test, pred)

f1 = f1_score(y_test, pred)

matriz = confusion_matrix(y_test, pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print(matriz)

Accuracy: 0.5054945054945055
Precision: 0.5084745762711864
Recall: 0.6521739130434783
F1: 0.5714285714285714
[[16 29]
 [16 30]]


CELDA 13 — Señales

In [14]:
senales = [
    "BUY" if x == 1 else "SELL"
    for x in modelo.predict(X_scaled)
]

CELDA 14 — JSON

In [15]:
salida = {
    "fechas": df.index.strftime("%Y-%m-%d").tolist(),

    "precios": df["Close"].round(4).tolist(),

    "senales": senales,

    "senal": senales[-1],

    "conf": float(
        max(
            modelo.predict_proba(
                X_scaled[-1:].reshape(1,-1)
            )[0]
        )
    ) if hasattr(modelo, "predict_proba") else 0.75,

    "metricas": {
        "accuracy": round(float(accuracy),4),
        "precision": round(float(precision),4),
        "recall": round(float(recall),4),
        "f1": round(float(f1),4)
    },

    "matriz": matriz.tolist(),

    "kernel": {
        "tipo": grid.best_params_["kernel"],
        "C": grid.best_params_["C"],
        "gamma": str(grid.best_params_["gamma"])
    }
}

CELDA 15 — Exportación

In [16]:
with open("svc.json","w") as archivo:
    json.dump(
        salida,
        archivo,
        indent=4
    )

print("svc.json generado.")

svc.json generado.


CELDA 16 — Descarga

In [17]:
from google.colab import files

files.download("svc.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>